In [1]:
from dotenv import load_dotenv
from pathlib import Path
import os

cwd = Path.cwd()
env_path = cwd / ".env" if (cwd / ".env").exists() else cwd.parent / ".env"

print("env_path =", env_path)
print("exists =", env_path.exists())
print("loaded =", load_dotenv(env_path, override=True))
print("api_key =", repr(os.getenv("OPENAI_API_KEY")[:10] if os.getenv("OPENAI_API_KEY") else None))

env_path = c:\Users\USER\Desktop\complypilot-jb\.env
exists = True
loaded = True
api_key = 'sk-proj-61'


In [2]:
# [중요] 폴더 전체를 읽어오기 위한 라이브러리 추가
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

import os
import shutil
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

pdf_dir = PROJECT_ROOT / "data" / "vectordb"
db_path = PROJECT_ROOT / "data" / "chromadb"
db_path.mkdir(parents=True, exist_ok=True)

loader = DirectoryLoader(
    path=str(pdf_dir),
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
)

print("폴더 내 모든 PDF 파일 로드 시작...")
documents_pdf = loader.load()
print(f"총 {len(documents_pdf)} 개의 전체 페이지가 로드됨")

# 2. 텍스트 분할 (기존 코드와 동일)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 900,       # 조항이 통째로 들어가도록 크기를 900으로 확대
    chunk_overlap = 150,
    separators = ["\n\n", "\n", " ", ""]
)
split_docs = text_splitter.split_documents(documents_pdf)
print(f"분할된 총 청크 개수: {len(split_docs)}")

# 3. 임베딩 및 벡터스토어 생성 (기존 코드와 동일)
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_documents(
    documents=split_docs, 
    embedding=embedding_model,
    persist_directory=str(db_path)
)

print("모든 문서가 Vector DB에 성공적으로 저장되었습니다!")

C:\Users\USER\AppData\Local\Temp\ipykernel_11852\1003840607.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader


폴더 내 모든 PDF 파일 로드 시작...
총 186 개의 전체 페이지가 로드됨
분할된 총 청크 개수: 343
모든 문서가 Vector DB에 성공적으로 저장되었습니다!


In [3]:
from langchain_openai import ChatOpenAI
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_classic.memory import ConversationBufferMemory

# (1) 리트리버(Retriever) 생성
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 7})

# (2) GPT-4.1-mini 모델 설정
llm = ChatOpenAI(model_name="gpt-4o")

# (3) 메모리 추가 (대화 문맥 유지)
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True, output_key="answer")

# (4) RAG 기반 ConversationalRetrievalChain 구성
qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    return_source_documents=True  # 검색된 문서 출력 옵션
)

C:\Users\USER\AppData\Local\Temp\ipykernel_11852\2802484103.py:12: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True, output_key="answer")


In [4]:
query = "금융광고를 할 때 필수적으로 포함해야 하는 문구나 정보에는 어떤 것들이 있어?"
response = qa_chain({"question": query})

# (1) 챗봇의 최종 답변 출력
print("[최종 답변]")
print(response["answer"])
print("\n" + "="*50 + "\n")

# (2) 💡 챗봇이 답변을 만들기 위해 벡터 DB에서 찾아온 근거 문서들 출력
print("[벡터 DB가 찾아온 실제 PDF 근거 본문]")
for i, doc in enumerate(response.get("source_documents", [])):
    print(f"\n📄 [근거 문서 {i+1}]")
    # 어떤 PDF 파일의 몇 번째 페이지인지 출처 확인
    print(f"출처 파일: {doc.metadata.get('source')} (Page: {doc.metadata.get('page', 0) + 1})")
    print(f"매칭된 실제 문맥(Context):\n{doc.page_content[:400]}...") # 앞 400자만 출력
    print("-" * 30)

C:\Users\USER\AppData\Local\Temp\ipykernel_11852\1711612605.py:2: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  response = qa_chain({"question": query})


[최종 답변]
금융광고를 할 때 필수적으로 포함해야 하는 문구나 정보는 다음과 같습니다:

1. 계약 체결 전 설명서 및 약관을 읽어볼 것을 권유하는 내용
2. 금융상품판매업자 등의 명칭, 금융상품의 내용
3. 금융상품 유형별 필수 포함사항:
   - 보장성, 투자성, 예금성, 대출성 상품 각각에 따라 필요한 세부 정보
4. 법령 및 내부통제기준에 따른 광고 관련 절차의 준수 여부

이 외에도 특정 금융소비자 보호를 위해 대통령령으로 정하는 사항들도 포함되어야 할 수 있습니다.


[벡터 DB가 찾아온 실제 PDF 근거 본문]

📄 [근거 문서 1]
출처 파일: c:\Users\USER\Desktop\complypilot-jb\data\vectordb\별첨자료_금융광고규제가이드라인.pdf (Page: 14)
매칭된 실제 문맥(Context):
- 12 -
Ⅳ. 광고의 내용 및 방법
관련 법령 주요내용
광고 시 금소법 뿐만 아니라 표시광고법, 방송법, 대부업법 등 
다른 법령에 위배되는 사항이 있는지도 꼼꼼히 확인해야* 함
    * 금소법 제6조(다른 법률과의 관계) 금융소비자 보호에 관하여 다른 법률에서 
특별히 정한 경우를 제외하고는 이 법에서 정하는 바에 따른다
 ㅇ 특히 유튜브, 블로그 등 온라인 매체를 통한 광고 시 뒷광고*(hidden 
ad) 이슈가 발생하지 않도록 최근 공정위에서 개정한 「추천·
보증 등에 관한 표시·광고 심사지침」을 준수해야 할 것임
    * 유명인이 광고를 하면서 광고주와의 경제적 이해관계를 표시하지 않는 경우 등
금소법령상 광고 내용에 포함시키도록 열거된 사항은 광고의 
목적, 광고매체의 특성 등을 감안하...
------------------------------

📄 [근거 문서 2]
출처 파일: c:\Users\USER\Desktop\complypilot-jb\data\vectordb\별첨자료_금융광고규제가이드라인.pdf (Page: 5)
매칭된 실제 문맥(Context):
- 3 -
관련 주요 질의․답변
1. 협

In [5]:
query = query = "대출 광고에서 '최저금리', '누구나 승인', '수수료 무료' 같은 표현은 왜 주의해야 해?"
response = qa_chain({"question": query})

# (1) 챗봇의 최종 답변 출력
print("[최종 답변]")
print(response["answer"])
print("\n" + "="*50 + "\n")

# (2) 💡 챗봇이 답변을 만들기 위해 벡터 DB에서 찾아온 근거 문서들 출력
print("[벡터 DB가 찾아온 실제 PDF 근거 본문]")
for i, doc in enumerate(response.get("source_documents", [])):
    print(f"\n📄 [근거 문서 {i+1}]")
    # 어떤 PDF 파일의 몇 번째 페이지인지 출처 확인
    print(f"출처 파일: {doc.metadata.get('source')} (Page: {doc.metadata.get('page', 0) + 1})")
    print(f"매칭된 실제 문맥(Context):\n{doc.page_content[:400]}...") # 앞 400자만 출력
    print("-" * 30)

[최종 답변]
대출 광고에서 '최저금리', '누구나 승인', '수수료 무료'와 같은 표현은 소비자가 이를 오해할 수 있는 가능성이 있기 때문에 주의해야 합니다. 

1. **'최저금리'**: 이러한 표현은 소비자가 최저금리가 자주 제공되는 조건이라고 오인하게 만들 수 있으며, 금소법에 따라 이자율의 범위 및 산출기준을 명확히 제시하여 소비자가 오해하지 않도록 해야 합니다.

2. **'누구나 승인'**: 이 표현은 모든 신청자가 대출 승인을 받을 수 있다고 오인하게 하며, 이는 잘못된 정보를 제공하는 것이 되기 때문에 금소법령상 금지되고 있습니다. 대출 승인은 보통 특정 신용 수준 등의 조건을 충족해야 승인될 수 있기 때문입니다.

3. **'수수료 무료'**: 광고에서 이러한 표현을 사용할 때는 실제로 소비자에게 적용되는 모든 요금과 수수료를 명확히 알리지 않을 경우 오해를 일으킬 수 있습니다. 이는 상담 시 명확한 정보를 제공해야 하는 이유입니다.

따라서 이러한 표현을 사용할 때는 오해를 방지하기 위해 명확하고 정확한 정보를 제공해야 합니다.


[벡터 DB가 찾아온 실제 PDF 근거 본문]

📄 [근거 문서 1]
출처 파일: c:\Users\USER\Desktop\complypilot-jb\data\vectordb\별첨자료_금융광고규제가이드라인.pdf (Page: 5)
매칭된 실제 문맥(Context):
- 3 -
관련 주요 질의․답변
1. 협회의 금융상품 정보 비교공시 서비스가 금소법상 광고에 
해당하는지?
□협회의 금융상품 정보 비교공시 서비스는 금소법에 따라 공익 
목적으로 제공된다는 점에서 광고로 보기 어려움
2. 금융정보 제공 방송도 금소법상 광고에 해당하는지?
□특정 금융상품판매업자의 금융상품에 관한 정보를 직·간접적
으로 제공하는 방송은 “금융상품 광고”로 볼 수 있음
 ㅇ 다만, 판매의도 없이 소비자가 금융상품판매업자나 금융상품을 
쉽게 유추할 수 없도록 조치(예: “A社”로 익명처리)하여 금융
정보를 제공하는 경우에는 광고로 보기 어려움